In [ ]:
#%load_ext autoreload
#%autoreload 2



from pathlib import Path
from src.kb_engine.rag_engine import RagEngine
from src.kb_engine.ingest import IngestionHandler

In [ ]:
import tempfile
persist_dir = Path("data/rag_engine_test")
rag_engine = RagEngine(
            name="anchor_rag3_test",
            persist_dir=persist_dir,
            ingestion_handler=IngestionHandler(),
            embedding_model="text-embedding-3-large",
        )
rag_engine.ingest(files=[Path("data/uploads/Alfa Laval LKH.pdf")])



In [ ]:
result = rag_engine.query("what is the connection dimension for Union SMS fro LKH-5?")
from devtools import debug
debug(result)

In [ ]:
from llama_index.core.base.response.schema import RESPONSE_TYPE,  Response
from src.kb_engine.patched_node_with_score import NodeWithScore

from pydantic import BaseModel
class ChunkInfo(BaseModel):
    file_path: str
    file_name: str
    page_nr: int

    def from_node(node: NodeWithScore):
        print(node.metadata)
        return ChunkInfo(
            file_path=node.metadata.get("filepath"),
            file_name=node.metadata.get("origin",{}).get("filename"),
            page_nr = node.metadata.get("doc_items", [])[0].get("prov", {})[0].get("page_no")
            
        )

    def get_page_image(self):
        file = Path(self.file_path)
        page_nr = self.page_nr
from devtools import debug
debug(result.source_nodes[0].metadata.get("doc_items", [])[0].get("prov", {})[0].get("page_no"))


In [ ]:

for node in result.source_nodes:
    
    print("--------------------------------")

    debug(ChunkInfo.from_node(node))
    print("--------------------------------")



In [ ]:
[
    {
        'self_ref': '#/texts/12',
        'parent': {
            '$ref': '#/body'
        },
        'children': [],
        'content_layer': 'body',
        'label': 'text',
        'prov': [
            {
                'page_no': 4,
                'bbox': {
                    'l': 78.0,
                    't': 527.0699705078125,
                    'r': 553.0,
                    'b': 493.81997050781246,
                    'coord_origin': 'BOTTOMLEFT'
                },
                'charspan': [
                    0,
                    203
                ]
            }
        ]
    }
]

In [ ]:
from pprint import pprint
from IPython.display import display
                                                                                                                                                              
def extract_page_bboxes(resp):
    rows = []                                                                                                                                               
    for rank, n in enumerate(resp.source_nodes, start=1):                                                                                                   
        md = n.metadata or {}                                                                                                                               
        for item in md.get("doc_items", []):                                                                                                                
            for prov in item.get("prov", []):                                                                                                               
                rows.append({                                                                                                                               
                    "rank": rank,                                                                                                                           
                    "score": float(n.score or 0.0),                                                                                                         
                    "file": md.get("filepath"),                                                                                                             
                    "heading": (md.get("headings") or [None])[0],                                                                                           
                    "label": item.get("label"),                                                                                                             
                    "page_no": prov.get("page_no"),                                                                                                         
                    "bbox": prov.get("bbox"),                                                                                                               
                })                                                                                                                                          
    return rows
                                                                                                                                                              
hits = extract_page_bboxes(result)                                                                                                                          
pprint(hits[:5])  # page + bbox values 

In [ ]:
display(result.source_nodes[0].to_image(scale=2)) 

In [ ]:
# 1) Re-ingest after code change                                                                                                                            
from pathlib import Path                                                                                                                                    
rag_engine.ingest(files=[Path("../uploads/Alfa Laval LKH.pdf")])     

In [ ]:
# 2) Query                                                                                                                                                  
result = rag_engine.query("what is the connection dimension for Union SMS for LKH-5?") 

In [ ]:
# 3) Find intersection cell (row label + column label)                                                                                                      
def find_table_cell(result, row_hint: str, col_hint: str):                                                                                                  
    row_hint = row_hint.lower()                                                                                                                             
    col_hint = col_hint.lower()                                                                                                                             
                                                                                                                                                            
    for node in result.source_nodes:                                                                                                                        
        cells = node.metadata.get("table_cells", [])                                                                                                        
        if not cells:                                                                                                                                       
            continue                                                                                                                                        
                                                                                                                                                            
        row_ids = {                                                                                                                                         
            c.get("row_start")                                                                                                                              
            for c in cells                                                                                                                                  
            if row_hint in str(c.get("text", "")).lower()                                                                                                   
        }                                                                                                                                                   
        col_ids = {                                                                                                                                         
            c.get("col_start")                                                                                                                              
            for c in cells                                                                                                                                  
            if col_hint in str(c.get("text", "")).lower()                                                                                                   
        }                                                                                                                                                   
                                                                                                                                                            
        for c in cells:                                                                                                                                     
            if c.get("row_start") in row_ids and c.get("col_start") in col_ids:                                                                             
                return node, c                                                                                                                              
                                                                                                                                                            
    return None, None 

In [ ]:
node, cell = find_table_cell(result, row_hint="LKH-5", col_hint="Union SMS")                                                                

In [ ]:
print(cell)